# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [2]:
import torch
import math

In [23]:
# ✏️ YOUR IMPLEMENTATION HERE
#这个就是需要把上面的给计算出来就行
#首先是这个d_k应该怎么算出来，是k的最后一个维度
def scaled_dot_product_attention(Q, K, V):
    d_k= K.size(-1)
    scores =Q@ K.transpose(-2,-1) / math.sqrt(d_k)
    return torch.softmax(scores,dim = -1)@V

In [24]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


In [25]:
# ✅ SUBMIT
from torch_judge import check
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.0ms)
  ✅ [2/4] Numerical correctness (0.6ms)
  ✅ [3/4] Gradient check (3.1ms)
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (4.9ms total)
  Progress saved. Run status() to see your dashboard.



交叉注意力，自注意力有什么区别

在公式上是没有什么区别，区别在于数据的来源上，即QKV是从哪来的，

对于自注意力机制来言，QKV是同一个X乘不同的权重得来的，即我们的decoder only就用这个

交叉注意力，Q来自序列A，KV来自序列B，一般用在那个encoder和decoder模型，用或者多模态中，用文本Q去查询图像的KV等

In [11]:
print(Q@K.T)

RuntimeError: The size of tensor a (2) must match the size of tensor b (8) at non-singleton dimension 0

In [ ]:
#这里注意一下，直接乘是不行的，因为第一个维度是batch，我们是要第一个维度不变，然后让后面两个维度做标准的矩阵乘法
#在pytorch中，*就是两个tensor按照对应位置的元素相乘，@是标准的矩阵乘法，此外torch.bmm是两个标准的矩阵（可以带有batch，即维度可以是batch，1，2和batch 2，1
#bmm不支持广播机制，*和@支持
print(torch.matmul(Q,K.T))

RuntimeError: The size of tensor a (2) must match the size of tensor b (8) at non-singleton dimension 0

广播机制是 PyTorch（以及 NumPy）中用于处理形状不同但满足特定兼容性条件的张量进行数学运算的一套底层规则。

从工程与内存的角度来看，当两个形状不同的张量进行运算时，广播机制会在逻辑上（而非物理内存中）将较小的张量扩展至与较大张量相同的形状，从而使逐元素操作或批量运算成为可能。这一机制避免了显式的数据复制（如调用 torch.repeat），极大地节省了显存/内存开销并提升了计算图的执行效率。

PyTorch 判断两个张量是否可以触发广播机制，严格遵循从右向左（Right-to-Left）的尾部维度对齐原则。

假设有两个张量 A 和 B，系统从它们形状的最后一个维度开始向前逐个比较。两个维度满足以下三个条件之一，即被视为兼容（Compatible）：

维度大小完全相等。

其中一个维度的大小为 1（系统会将大小为 1 的维度在逻辑上复制，扩展至与另一个张量该维度相同的大小）。

其中一个张量在该位置没有维度（即该张量维数较低。系统会在其形状的最左侧隐式添加大小为 1 的维度，随后继续应用规则 2）。

In [15]:
#矩阵转置，这里有三个维度，直接.T是不行的，应该用transpose
print(K.transpose(-1,-2) == K.transpose(-2,-1))

tensor([[[True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True]],

        [[True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True],
         [True, True, True, True]]])


在计算缩放点积时，注意力分数矩阵的形状为 (batch_size, num_heads, query_len, key_len)。注意力机制的严格数学定义要求：针对每一个特定的 Query 向量，其分配给所有 Key 向量的注意力权重标量之和必须等于 1.0。因此，归一化操作必须严格沿着 key_len 维度执行（通常为 dim=-1）